# Ship of Theseus corpus: lexical vs semantic decay, style vs content

Implements `docs/sot_analysis_plan.md` on the published English Ship of Theseus corpus (Tripto et al., 2024). Nothing is generated here: every text is from the corpus. This notebook produces the three figures in the final report.

1. **Data:** 7 datasets, 7 sources, 19,342 originals (T0), and 7 paraphraser variants × 3 chained rounds (T1–T3).
2. **Lexical vs semantic metrics:** BLEU, ROUGE-L and word retention (the "planks") vs BERTScore and a sentence-embedding cosine (the "hull"), T0 → T1/T2/T3, for ChatGPT, PaLM, Dipper and Pegasus (`decay_curves.png`).
3. **Style vs content decay:** does the meaning stay intact while the words and the style are replaced? (`style_vs_content.png`, `style_blocks_and_sources.png`)

In [ ]:
import shutil
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pilot.config import ROOT, load_config, set_seeds
from pilot.metrics import DATASETS, bootstrap_mean_ci, load_corpus, load_reference

warnings.filterwarnings("ignore", category=FutureWarning)
cfg = load_config()
set_seeds(cfg["seed"])
SOT, DERIVED = cfg["paths"]["sot"], cfg["paths"]["derived"]
OUT = cfg["paths"]["results"] / "sot_analysis"
OUT.mkdir(parents=True, exist_ok=True)

# Paraphrasers: the four primary variants, plus the intensity variants of Dipper and Pegasus.
PRIMARY = ["chatgpt", "palm", "dipper", "pegasus(full)"]
LABEL = {"chatgpt": "ChatGPT", "palm": "PaLM", "dipper": "Dipper", "pegasus(full)": "Pegasus",
         "dipper(low)": "Dipper (low)", "dipper(high)": "Dipper (high)", "pegasus(slight)": "Pegasus (slight)"}

# Categorical palette (validated: adjacent CVD ΔE 9.1, normal-vision 22.9; aqua/yellow < 3:1 contrast,
# so every chart carries direct labels and a CSV table). Color follows the paraphraser, in fixed order.
COLOR = {"chatgpt": "#2a78d6", "palm": "#eb6834", "dipper": "#1baf7a", "pegasus(full)": "#eda100"}
MARKER = {"chatgpt": "o", "palm": "s", "dipper": "^", "pegasus(full)": "D"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e1"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight", "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.labelsize": 9, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "lines.linewidth": 2, "lines.markersize": 6,
    "legend.frameon": False, "legend.fontsize": 8, "font.size": 9,
})

def label_ends(ax, items, x, min_gap=None, fmt="{label} {y:.2f}"):
    """Direct labels at line ends, nudged apart vertically so they never collide.
    items: list of (label, y, color-or-None)."""
    lo, hi = ax.get_ylim()
    gap = min_gap if min_gap is not None else 0.045 * (hi - lo)
    placed = []
    for label, y, _ in sorted(items, key=lambda t: t[1]):
        pos = max(y, placed[-1] + gap) if placed else y
        placed.append(pos)
        ax.annotate(fmt.format(label=label, y=y), (x, y), xytext=(x + 0.12, pos), textcoords="data",
                    va="center", fontsize=8, color=INK,
                    arrowprops=dict(arrowstyle="-", color=GRID, linewidth=0.6) if abs(pos - y) > 1e-9 else None)


def save(fig, name, table=None):
    fig.savefig(OUT / f"{name}.png")
    if table is not None:
        table.to_csv(OUT / f"{name}.csv", index=False)

print("results ->", OUT.relative_to(ROOT))

## 1. Data

All 7 datasets are copied from the upstream clone into git-ignored `data/sot/` (read-only), then loaded as one long table with one row per text: dataset, source, key, variant, round (0 = T0) and text. `train_datasets/` (never paraphrased, disjoint keys) is loaded as the **reference set**, used for normalization floors and for fitting style features.

In [ ]:
for ds in DATASETS:
    for rel in (f"paraphrased_datasets/{ds}_paraphrased.csv", f"train_datasets/{ds}_train.csv"):
        dst = SOT / rel
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(cfg["paths"]["corpus_src"] / rel, dst)
            dst.chmod(0o444)

corpus = load_corpus(SOT)
reference = load_reference(SOT)
corpus["words"] = corpus.text.str.split().str.len()
reference["words"] = reference.text.str.split().str.len()

originals = corpus[corpus["round"] == 0]
summary = pd.DataFrame({
    "originals": originals.groupby("dataset").size(),
    "keys": originals.groupby("dataset").key.nunique(),
    "paraphrased_rows": corpus[corpus["round"] > 0].groupby("dataset").size(),
    "reference_texts": reference.groupby("dataset").size(),
    "median_T0_words": originals.groupby("dataset").words.median(),
})
summary.loc["total"] = [summary.originals.sum(), summary["keys"].sum(), summary.paraphrased_rows.sum(),
                        summary.reference_texts.sum(), originals.words.median()]
print(f"{len(corpus):,} corpus rows | {corpus.source.nunique()} sources | variants: {sorted(corpus.variant.unique())}")
summary.astype(int)

In [ ]:
ID = ["dataset", "source", "key"]
chains = corpus[corpus["round"] > 0]

## 2. Lexical vs semantic metrics

**Scope.** PaLM chains are often incomplete (only 87.5% of originals reach T3), so the comparison uses **completer originals only**: those that all four primary paraphrasers (ChatGPT, PaLM, Dipper, Pegasus-full) carry to T3, and that have a T0 text. Every paraphraser is then scored on exactly the same texts. `SCOPE = "full"` keeps all 16,682 of them; `SCOPE = "sample"` draws 60 keys per dataset (seeded) and keeps every completer original of those keys.

At full scope the slow scores come from background scripts (`scripts/score_lexical.py full`, `scripts/score_semantic.py`, `scripts/style_features.py`), which write the caches the cells below read.

**Pairs.**
- *Cumulative:* T0 vs Tn, n = 1, 2, 3 (the decay curve).
- *Step:* Tn−1 vs Tn, n = 2, 3 (how much each extra round changes; for n = 1, step = cumulative).
- *Floor:* 200 unrelated **T0** pairs per dataset (two originals from different keys).

**Metrics (team standard, `docs/metric_standard.md`).** Word retention counts content words only (stop words excluded). BERTScore is baseline-rescaled. MPNet cosine embeds long texts as a token-weighted mean of sentence-packed 384-token chunks.

Scores are cached to `data/derived/sot_pairs_*.parquet`.

In [ ]:
# SCOPE = "sample": 60 keys per dataset (fast; the original analysis).
# SCOPE = "full":   every completer original (the slow parts run as background scripts; see §2 notes).
SCOPE = "full"
N_KEYS = 60
VARIANTS = PRIMARY + ["dipper(low)", "dipper(high)", "pegasus(slight)"]
KEYS = ID + ["variant", "round", "kind"]
texts = corpus.set_index(ID + ["variant", "round"]).text  # (dataset, source, key, variant, round) -> text

reach_t3 = chains[(chains["round"] == 3) & chains.variant.isin(PRIMARY)].groupby(ID).variant.nunique()
completers = reach_t3[reach_t3 == len(PRIMARY)].reset_index()[ID]
# One chain (xsum / OpenAI / xsum-492) has all its paraphrases but no T0 text in the corpus; nothing to compare against.
completers = completers.merge(corpus.loc[corpus["round"] == 0, ID], on=ID)
if SCOPE == "full":
    sample = completers.copy()
else:
    rng = np.random.default_rng([cfg["seed"], 31])
    sample_keys = pd.concat([
        part.drop_duplicates("key").sample(N_KEYS, random_state=int(rng.integers(1e9)))[["dataset", "key"]]
        for _, part in completers.groupby("dataset")
    ])
    sample = completers.merge(sample_keys, on=["dataset", "key"])

# Pairs (cumulative T0 vs Tn, step Tn-1 vs Tn), built by the same function the background scripts use.
from pilot.metrics import build_pairs, t0_floor_pairs
pairs = build_pairs(corpus, sample, VARIANTS)

# Floor (team standard): 200 unrelated T0 pairs per dataset, two originals from different keys (articles).
floors = t0_floor_pairs(corpus, 200, cfg["seed"])

# The background scoring scripts (scripts/score_semantic.py, scripts/style_features.py) read these.
SAMPLE_PATH, FLOORS_PATH = DERIVED / "sot_sample.parquet", DERIVED / "sot_floors.parquet"
sample.to_parquet(SAMPLE_PATH, index=False)
floors.to_parquet(FLOORS_PATH, index=False)

print(f"SCOPE={SCOPE} | completer originals: {len(completers):,} of {len(originals):,} | analysed: {len(sample):,} originals "
      f"on {sample[['dataset', 'key']].drop_duplicates().shape[0]:,} keys | pairs: {len(pairs):,} | floor pairs: {len(floors)}")
sample.groupby(["dataset"]).size().to_frame("originals").T

In [ ]:
import time

from pilot.metrics import score_lexical_chunked

LEX_CACHE = DERIVED / f"sot_pairs_lexical_{SCOPE}.parquet"  # one file per scope
FLOOR_LEX_CACHE = DERIVED / "sot_floors_lexical_t0.parquet"  # floors = unrelated T0 pairs (team standard)

# Chunked, bounded-memory scoring: feeding all 580k full-scope pairs to a process pool at once
# exhausted memory and killed the kernel. SCOPE="full" comes from `python scripts/score_lexical.py`.
started = time.time()
if not LEX_CACHE.exists():
    if SCOPE == "full":
        raise RuntimeError("run `python scripts/score_lexical.py full` in the background, then re-run this cell")
    score_lexical_chunked(pairs, LEX_CACHE)
if not FLOOR_LEX_CACHE.exists():
    score_lexical_chunked(floors.assign(source="", key="", variant="", round=0, kind="floor"), FLOOR_LEX_CACHE)
lex = pd.read_parquet(LEX_CACHE)
floor_lex = pd.read_parquet(FLOOR_LEX_CACHE)
assert len(lex) == len(pairs) and len(floor_lex) == len(floors)
print(f"lexical scores: {len(lex):,} pairs + {len(floor_lex):,} floor pairs in {time.time() - started:.0f}s")
# word_retention = content-word (stop words excluded) type recall, per the team standard.
LEXICAL = ["bleu", "rouge1", "rouge2", "rougeL", "word_retention"]
lex[lex.kind == "cumulative"].groupby(["variant", "round"])[LEXICAL].mean().round(3)

In [ ]:
import importlib

import pilot.metrics
importlib.reload(pilot.metrics)
from pilot.metrics import chain_jobs, score_semantic

# v2 = team standard: rescaled BERTScore + MPNet cosine as a token-weighted mean of sentence-packed chunks.
SEM_CACHE, FLOOR_SEM_CACHE = DERIVED / "sot_pairs_semantic_v2.parquet", DERIVED / "sot_floors_semantic_v2.parquet"
SEMANTIC = ["bertscore_f1_rescaled", "bertscore_p_rescaled", "bertscore_r_rescaled", "mpnet_cosine"]

def floor_jobs(frame):
    for r in frame.itertuples(index=False):
        yield {"dataset": r.dataset, "floor_id": r.floor_id}, [r.ref, r.hyp], [(0, 1, {"kind": "floor"})]

# The cache is keyed by original and resumable, and a full-corpus cache also serves the sample.
# SCOPE="sample" computes missing scores here (about 1-2 h on MPS);
# SCOPE="full" (about 9 h) must come from `python scripts/score_semantic.py`, never from the kernel.
def semantic_missing():
    if not (SEM_CACHE.exists() and FLOOR_SEM_CACHE.exists()):
        return len(sample)
    cached = pd.read_parquet(SEM_CACHE, columns=ID).drop_duplicates()
    return len(sample.merge(cached, on=ID, how="left", indicator=True).query("_merge == 'left_only'"))

missing = semantic_missing()
if missing and SCOPE == "full":
    raise RuntimeError(f"{missing:,} originals lack semantic scores: run `python scripts/score_semantic.py` "
                       "in the background (resumable), then re-run from this cell")
if missing:
    score_semantic(chain_jobs(sample, texts, VARIANTS, ID), SEM_CACHE, ID)
    score_semantic(floor_jobs(floors), FLOOR_SEM_CACHE, ["dataset", "floor_id"])
sem = pd.read_parquet(SEM_CACHE).merge(sample, on=ID)  # only the originals in this scope
floor_sem = pd.read_parquet(FLOOR_SEM_CACHE)
assert len(floor_sem) == len(floors), "floor semantic cache is from another floor set; delete it and re-run"
print(f"semantic scores: {len(sem):,} pairs + {len(floor_sem):,} floor pairs")
sem[sem.kind == "cumulative"].groupby(["variant", "round"])[SEMANTIC].mean().round(3)

In [ ]:
import json

importlib.reload(pilot.metrics)
from pilot.metrics import STYLE_COMPONENTS, StyleScorer, extract_style, reference_subset, top_function_words, top_words

# Style = union of both team pipelines (see pilot.metrics.StyleScorer and STYLE_FAMILIES).
# v2 features add dependency-label and most-frequent-word rates (for dep_jsd and Burrows' Delta).
STYLE_CACHE, VOCAB_PATH = DERIVED / "sot_style_features_v2.parquet", DERIVED / "sot_style_vocab.json"
# Per-pair style scores take a while at full scope, so they are cached like the lexical scores.
STYLE_SCORES = DERIVED / f"sot_pairs_style_v2_{SCOPE}.parquet"
FLOOR_STYLE_SCORES = DERIVED / "sot_floors_style_v2.parquet"


def compute_style_scores():
    # The reference subset (train_datasets/, never paraphrased) fixes the function-word and MFW lists, the
    # z-scoring and the char n-gram vocabulary, so no rewritten output influences how style is measured.
    ref_sub = reference_subset(reference, 300, cfg["seed"])
    unique_texts = list(pd.unique(pd.concat([pairs.ref, pairs.hyp, floors.ref, floors.hyp, ref_sub.text])))

    # spaCy is slow; scripts/style_features.py computes the same cache with multiprocessing (which deadlocks
    # inside Jupyter on macOS). Without a cache, fall back to single-process extraction here.
    if STYLE_CACHE.exists():
        feats = pd.read_parquet(STYLE_CACHE)
    else:
        if SCOPE == "full":
            raise RuntimeError("run `python scripts/style_features.py` in the background, then re-run this cell")
        fw, mfw = top_function_words(ref_sub.text), top_words(ref_sub.text)
        feats = extract_style(unique_texts, fw, mfw)
        feats.to_parquet(STYLE_CACHE, index=False)
        VOCAB_PATH.write_text(json.dumps({"function_words": fw, "mfw": mfw}))
    scorer = StyleScorer(feats, ref_sub.text.tolist())
    # Keys come from `pairs` itself (row-aligned with the scores), never from another cache's row order.
    p = pairs.reset_index(drop=True)
    style = pd.concat([p[KEYS], scorer.score(p.ref, p.hyp)], axis=1)
    floor_style = pd.concat([floors[["dataset", "floor_id"]].reset_index(drop=True),
                             scorer.score(floors.ref.reset_index(drop=True), floors.hyp.reset_index(drop=True))], axis=1)
    return style, floor_style


if STYLE_SCORES.exists() and FLOOR_STYLE_SCORES.exists():
    style, floor_style = pd.read_parquet(STYLE_SCORES), pd.read_parquet(FLOOR_STYLE_SCORES)
    # The cache must cover exactly the current pairs (it is merged on keys downstream, never by position).
    assert len(style) == len(pairs) and style[KEYS].merge(pairs[KEYS]).shape[0] == len(pairs), "stale style cache; delete it"
    assert len(floor_style) == len(floors), "stale floor style cache; delete it"
else:
    style, floor_style = compute_style_scores()
    style.to_parquet(STYLE_SCORES, index=False); floor_style.to_parquet(FLOOR_STYLE_SCORES, index=False)
print(f"style scores: {len(style):,} pairs + {len(floor_style):,} floor pairs | components: {STYLE_COMPONENTS}")
style[style.kind == "cumulative"].groupby(["variant", "round"])[STYLE_COMPONENTS].mean().round(3)

### 2.1 One scale for every metric: retained share

Each metric is normalized per dataset against its **floor**: the mean score of 200 **unrelated T0 pairs** from that dataset (two originals from different keys). This follows the team metric standard (`docs/metric_standard.md`).

For similarities (BLEU, ROUGE, word retention, rescaled BERTScore, cosines), which score 1 for identical texts:
$$\text{retained} = \frac{s - \text{floor}}{1 - \text{floor}}$$
For distances (punctuation L1, POS/dependency JSD, Burrows' Delta), which score 0 for identical texts:
$$\text{retained} = \frac{\text{floor} - d}{\text{floor}}$$

Either way, **retained = 1 means identical to T0 and 0 means no closer than an unrelated original from the same dataset**. Lexical, semantic and style decay can then be compared on one axis, which raw scores don't allow.

**Combined style score.** The 11 style components (the union of both team pipelines) are grouped into five families: surface, punctuation, function words, syntax and character n-grams. Each family score is the mean of its components' retained scores, and `style` is the mean of the five family scores. Overlapping measures therefore don't get double weight.

In [ ]:
from pilot.metrics import STYLE_FAMILIES, add_style_composite, retained

KEYS = ID + ["variant", "round", "kind"]
STYLE = STYLE_COMPONENTS
METRICS = LEXICAL + SEMANTIC + STYLE          # raw scores; each gets a *_ret column below
FAMILIES = [f"style_fam_{f}" for f in STYLE_FAMILIES]
RET_METRICS = METRICS + FAMILIES + ["style"]  # everything on the retained scale (style = combined score)
NAME = {"bleu": "BLEU", "rouge1": "ROUGE-1", "rouge2": "ROUGE-2", "rougeL": "ROUGE-L",
        "word_retention": "Word retention (content words)",
        "bertscore_f1_rescaled": "BERTScore F1 (rescaled)", "bertscore_p_rescaled": "BERTScore precision (rescaled)",
        "bertscore_r_rescaled": "BERTScore recall (rescaled)", "mpnet_cosine": "MPNet cosine",
        "style": "Style (combined)",
        "style_surface": "Surface (z-cosine)", "style_punctuation": "Punctuation (z-cosine)", "punct_l1": "Punctuation L1",
        "style_function_words": "Function words (z-cosine)", "fw_cosine": "Function-word cosine",
        "burrows_delta": "Burrows' Delta", "style_syntax_pos": "POS (z-cosine)", "pos_jsd": "POS JSD",
        "dep_jsd": "Dependency JSD", "style_charngram": "Char 2-4-gram TF-IDF", "char3_cosine": "Char 3-gram cosine",
        **{f"style_fam_{f}": f"Style: {f.replace('_', ' ')}" for f in STYLE_FAMILIES}}

scores = (lex.merge(sem, on=KEYS, how="left", validate="1:1")
             .merge(style, on=KEYS, how="left", validate="1:1"))
assert len(scores) == len(lex), "pairs lost in the merge"
missing = scores[METRICS].isna().sum()
# Undefined, not missing: word retention when the reference has no content words, BERTScore precision
# when the output is empty, style distributions of an empty text (degenerate outputs). Left as NaN.
assert missing.sum() < 0.001 * len(scores) * len(METRICS), f"too many undefined scores: {missing[missing > 0].to_dict()}"
print("undefined scores (degenerate outputs, left as NaN):", missing[missing > 0].to_dict())
p = pairs.reset_index(drop=True)
scores = scores.merge(p.assign(len_ratio=p.hyp.str.split().str.len() / p.ref.str.split().str.len())[KEYS + ["len_ratio"]],
                      on=KEYS, validate="1:1")

# Floors: mean score of the unrelated T0 pairs of each dataset.
floor_all = (floor_lex.assign(floor_id=floors.floor_id.to_numpy())[["dataset", "floor_id"] + LEXICAL]
             .merge(floor_sem, on=["dataset", "floor_id"], validate="1:1")
             .merge(floor_style, on=["dataset", "floor_id"], validate="1:1"))
floor = floor_all.groupby("dataset")[METRICS].mean()
for m in METRICS:
    scores[f"{m}_ret"] = retained(scores[m], scores.dataset.map(floor[m]), m)
scores = add_style_composite(scores)

floor.round(3).to_csv(OUT / "floors_by_dataset.csv")
print(f"{len(scores):,} scored pairs | floors (mean over datasets):")
floor.mean().round(3).to_frame("floor").T

### 2.2 Decay curves: planks vs hull

Mean retained share at T0–T3 for the four paraphrasers, with 95% bootstrap CIs (1,000 resamples of keys). The top row is lexical (the planks), the bottom row semantic and style. The table behind the figure is `decay_curves.csv`.

In [ ]:
cum = scores[scores.kind == "cumulative"]
CURVE_METRICS = ["bleu", "rougeL", "word_retention", "bertscore_f1_rescaled", "mpnet_cosine", "style"]
curves = pd.concat([bootstrap_mean_ci(cum, ["variant", "round"], f"{m}_ret").assign(metric=m) for m in RET_METRICS],
                   ignore_index=True)
curves.to_csv(OUT / "decay_curves.csv", index=False)

def plot_curves(ax, metric, variants, colors, markers=None, styles=None):
    for v in variants:
        d = curves[(curves.metric == metric) & (curves.variant == v)].sort_values("round")
        ax.fill_between([0, *d["round"]], [1, *d.ci_low], [1, *d.ci_high], color=colors[v], alpha=0.15, linewidth=0)
        ax.plot([0, *d["round"]], [1, *d["mean"]], color=colors[v], marker=(markers or {}).get(v, "o"),
                linestyle=(styles or {}).get(v, "-"), label=LABEL[v])
    ax.set_xticks([0, 1, 2, 3], ["T0", "T1", "T2", "T3"]); ax.set_title(NAME[metric], loc="left")
    ax.axhline(0, color=MUTED, linewidth=0.8, linestyle=":")

PANEL_FAMILY = {"bleu": "Lexical", "rougeL": "Lexical", "word_retention": "Lexical",
                "bertscore_f1_rescaled": "Semantic", "mpnet_cosine": "Semantic", "style": "Style"}
fig, axes = plt.subplots(2, 3, figsize=(12, 6.8), sharey=True)
for ax, m in zip(axes.flat, CURVE_METRICS):
    plot_curves(ax, m, PRIMARY, COLOR, MARKER)
    ax.set_title(f"{PANEL_FAMILY[m]} · {NAME[m]}", loc="left")
for ax in axes[:, 0]:
    ax.set_ylabel("retained (1 = T0, 0 = unrelated T0 text)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.01, 0.955), ncol=4)
fig.suptitle("Decay of words, meaning and style, T0 → T3 (95% CI)", x=0.01, ha="left",
             fontsize=13, color=INK, y=1.0)
fig.tight_layout(rect=(0, 0, 1, 0.92))
save(fig, "decay_curves")
plt.show()
(curves[curves.metric.isin(CURVE_METRICS) & curves.variant.isin(PRIMARY) & (curves["round"] == 3)]
 .pivot(index="variant", columns="metric", values="mean")[CURVE_METRICS].round(3))

## 3. Style vs content decay: does the hull hold while the planks change?

On the retained scale, **content** = rescaled BERTScore F1 and **style** = the combined style score (§2.1): the mean of five family scores (surface, punctuation, function words, syntax, character n-grams), each averaging its components. Feature z-scores, the function-word and most-frequent-word lists and the char n-gram vocabulary are fitted on held-out reference texts.

- **3.1** Style vs content vs words per paraphraser, and the decay gap (content − style) with CIs.
- **3.2** Which style families and components go first, and does Human style decay differently from LLM style?

In [ ]:
# 3.1 Style vs content per paraphraser, and the decay gap (content - style).
FAM_COLOR = {"bertscore_f1_rescaled": "#2a78d6", "style": "#eb6834", "word_retention": "#1baf7a"}  # 3 slots: valid all-pairs
FAM_LABEL = {"bertscore_f1_rescaled": "Content (BERTScore)", "style": "Style (combined)", "word_retention": "Words (retention)"}

fig, axes = plt.subplots(1, 4, figsize=(13, 3.8), sharey=True)
for ax, v in zip(axes, PRIMARY):
    ends = []
    for m, c in FAM_COLOR.items():
        d = curves[(curves.metric == m) & (curves.variant == v)].sort_values("round")
        ax.fill_between([0, *d["round"]], [1, *d.ci_low], [1, *d.ci_high], color=c, alpha=0.15, linewidth=0)
        ax.plot([0, *d["round"]], [1, *d["mean"]], color=c, marker="o", label=FAM_LABEL[m])
        ends.append((FAM_LABEL[m].split(" ")[0], d["mean"].iloc[-1], c))
    label_ends(ax, ends, 3, min_gap=0.065)
    ax.set_xticks([0, 1, 2, 3], ["T0", "T1", "T2", "T3"]); ax.set_xlim(-0.1, 4.4); ax.set_title(LABEL[v], loc="left")
    ax.axhline(0, color=MUTED, linewidth=0.8, linestyle=":")
axes[0].set_ylabel("retained (1 = T0, 0 = unrelated T0 text)"); axes[0].legend(loc="lower left")
fig.suptitle("Style vs content vs words, per paraphraser (95% CI)", x=0.01, ha="left", fontsize=12, color=INK)
fig.tight_layout()

cum = cum.assign(decay_gap=cum.bertscore_f1_rescaled_ret - cum.style_ret)
decay_gap = bootstrap_mean_ci(cum[cum.variant.isin(PRIMARY)], ["variant", "round"], "decay_gap")
save(fig, "style_vs_content", decay_gap)
plt.show()
print("decay gap = retained content - retained style (mean [95% CI]):")
decay_gap.assign(ci=lambda d: d.apply(lambda r: f"{r['mean']:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)
                 ).pivot(index="variant", columns="round", values="ci")

In [ ]:
# 3.2 Which style features go first, and whose style decays fastest (caveat: LLM-authored originals
# begin with the Human text's opening words, their generation prompt).
t3_all = cum[(cum["round"] == 3) & cum.variant.isin(PRIMARY)]
blocks_t3 = t3_all.groupby("variant")[[f"{f}_ret" for f in FAMILIES]].mean().reindex(PRIMARY)
blocks_t3.columns = [f.replace("style_fam_", "").replace("_", " ") for f in FAMILIES]
components_t3 = t3_all.groupby("variant")[[f"{m}_ret" for m in STYLE_COMPONENTS]].mean().reindex(PRIMARY)
components_t3.columns = [NAME[m] for m in STYLE_COMPONENTS]
by_source = t3_all.groupby(["source", "variant"]).style_ret.mean().unstack().reindex(columns=PRIMARY)
by_source = by_source.loc[by_source.mean(axis=1).sort_values().index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.1, 1]})
for ax, table, title in [(axes[0], blocks_t3, "Retained style at T3, by feature family"),
                         (axes[1], by_source, "Retained style (combined) at T3, by source")]:
    im = ax.imshow(table.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")  # sequential: magnitude
    ax.set_xticks(range(table.shape[1]), [LABEL.get(c, c) for c in table.columns], rotation=30, ha="right")
    ax.set_yticks(range(table.shape[0]), [LABEL.get(i, i) for i in table.index])
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            val = table.iat[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color="white" if val > 0.6 else INK)
    ax.grid(False); ax.set_title(title, loc="left")
fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="retained (1 = T0)")
save(fig, "style_blocks_and_sources", by_source.reset_index())
blocks_t3.round(3).to_csv(OUT / "style_blocks_T3.csv")
components_t3.round(3).to_csv(OUT / "style_components_T3.csv")
plt.show()
components_t3.round(3)